In [19]:
import warnings
warnings.filterwarnings("ignore")

### Importing the libraries

In [3]:
import sys
print(sys.executable)

/Users/macbook/Documents/Code/Machine-Learning-Projects/tf-env/bin/python


In [4]:
# basic + dates 
import numpy as np
import pandas as pd
from datetime import datetime

# data visualization
import matplotlib.pyplot as plt
import seaborn as sns # advanced vizs
%matplotlib inline

# statistics
from statsmodels.distributions.empirical_distribution import ECDF

# time series analysis
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# # prophet by Facebook
# from fbprophet import Prophet

### Geting the Data

In [5]:
train  = pd.read_csv("./dataset/train.csv")
store    = pd.read_csv("./dataset/store.csv")
test      = pd.read_csv("./dataset/test.csv")


train.head()

/var/folders/gk/q9d5jbtj33j94vb3kz2vgk4h0000gn/T/ipykernel_18257/1457875039.py:1: DtypeWarning: Columns (0: StateHoliday) have mixed types. Specify dtype option on import or set low_memory=False.
  train  = pd.read_csv("./dataset/train.csv")


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1


In [7]:
train.shape, store.shape, test.shape

((1017209, 9), (1115, 10), (41088, 8))

In [8]:
#check for missing values and data types
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 1017209 entries, 0 to 1017208
Data columns (total 9 columns):
 #   Column         Non-Null Count    Dtype 
---  ------         --------------    ----- 
 0   Store          1017209 non-null  int64 
 1   DayOfWeek      1017209 non-null  int64 
 2   Date           1017209 non-null  str   
 3   Sales          1017209 non-null  int64 
 4   Customers      1017209 non-null  int64 
 5   Open           1017209 non-null  int64 
 6   Promo          1017209 non-null  int64 
 7   StateHoliday   1017209 non-null  object
 8   SchoolHoliday  1017209 non-null  int64 
dtypes: int64(7), object(1), str(1)
memory usage: 69.8+ MB


In [9]:
store.info()

<class 'pandas.DataFrame'>
RangeIndex: 1115 entries, 0 to 1114
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Store                      1115 non-null   int64  
 1   StoreType                  1115 non-null   str    
 2   Assortment                 1115 non-null   str    
 3   CompetitionDistance        1112 non-null   float64
 4   CompetitionOpenSinceMonth  761 non-null    float64
 5   CompetitionOpenSinceYear   761 non-null    float64
 6   Promo2                     1115 non-null   int64  
 7   Promo2SinceWeek            571 non-null    float64
 8   Promo2SinceYear            571 non-null    float64
 9   PromoInterval              571 non-null    str    
dtypes: float64(5), int64(2), str(3)
memory usage: 87.2 KB


In [32]:
store.head(5)

,Store,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,StoreType_b,StoreType_c,StoreType_d,Assortment_b,Assortment_c
0,1,1270.0,9.0,2008.0,0,NaN,NaN,NaN,False,True,False,False,False
1,2,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct",False,False,False,False,False
2,3,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct",False,False,False,False,False
3,4,620.0,9.0,2009.0,0,NaN,NaN,NaN,False,True,False,False,True
4,5,29910.0,4.0,2015.0,0,NaN,NaN,NaN,False,False,False,False,False


In [12]:
#filling in the missing value in competition distance with -1
store['CompetitionDistance'] = store['CompetitionDistance'].fillna(-1, inplace=True)


/var/folders/gk/q9d5jbtj33j94vb3kz2vgk4h0000gn/T/ipykernel_18257/2223307206.py:2: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  store['CompetitionDistance'] = store['CompetitionDistance'].fillna(-1, inplace=True)


In [15]:
train = pd.merge(train, store, on='Store')

In [26]:
#feature engineering - adding new features day, month, year, day of week, week of year, is weekend
train['Month'] = train['Date'].dt.month
train['Year'] = train['Date'].dt.year


In [27]:
#feature engineering - hot encoding categorical variables
store = pd.get_dummies(store, columns=['StoreType', 'Assortment'], drop_first=True)

### Checking Data before going to the next step

In [ ]:
#all data types should be numeric for modelin 
train.shape
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 1017209 entries, 0 to 1017208
Data columns (total 39 columns):
 #   Column                       Non-Null Count    Dtype         
---  ------                       --------------    -----         
 0   Store                        1017209 non-null  int64         
 1   DayOfWeek                    1017209 non-null  int32         
 2   Date                         1017209 non-null  datetime64[us]
 3   Sales                        1017209 non-null  int64         
 4   Customers                    1017209 non-null  int64         
 5   Open                         1017209 non-null  int64         
 6   Promo                        1017209 non-null  int64         
 7   StateHoliday                 1017209 non-null  object        
 8   SchoolHoliday                1017209 non-null  int64         
 9   StoreType_x                  1017209 non-null  str           
 10  Assortment_x                 1017209 non-null  str           
 11  CompetitionDistance_x 

In [36]:
train.head(5)

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType_x,...,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,Day,Month,Year
0,1,4,2015-07-31,5263,555,1,1,0,1,c,...,1270.0,9.0,2008.0,0,NaN,NaN,NaN,31,7,2015
1,2,4,2015-07-31,6064,625,1,1,0,1,a,...,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct",31,7,2015
2,3,4,2015-07-31,8314,821,1,1,0,1,a,...,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct",31,7,2015
3,4,4,2015-07-31,13995,1498,1,1,0,1,c,...,620.0,9.0,2009.0,0,NaN,NaN,NaN,31,7,2015
4,5,4,2015-07-31,4822,559,1,1,0,1,a,...,29910.0,4.0,2015.0,0,NaN,NaN,NaN,31,7,2015


In [37]:
#fixing te Jupyter Notebook cell memory problem by saving the data to a CSV file

# 1. Reload the original data to clear the notebook memory
train = pd.read_csv('./dataset/train.csv')
store = pd.read_csv('./dataset/store.csv')

# 2. Clean and encode the store data
store['CompetitionOpenSinceMonth'].fillna(0, inplace=True)
store['CompetitionOpenSinceYear'].fillna(0, inplace=True)
store = pd.get_dummies(store, columns=['StoreType', 'Assortment'], drop_first=True)

# 3. Process the dates in the train data
train['Date'] = pd.to_datetime(train['Date'], format='%Y-%m-%d')
train['Month'] = train['Date'].dt.month
train['Year'] = train['Date'].dt.year
train['Day'] = train['Date'].dt.day

# 4. Do ONE final merge
train = pd.merge(train, store, on='Store')

/var/folders/gk/q9d5jbtj33j94vb3kz2vgk4h0000gn/T/ipykernel_18257/2560072948.py:4: DtypeWarning: Columns (0: StateHoliday) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv('./dataset/train.csv')
/var/folders/gk/q9d5jbtj33j94vb3kz2vgk4h0000gn/T/ipykernel_18257/2560072948.py:8: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stabl

In [39]:
#check the train data shape and info
train.shape
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 1017209 entries, 0 to 1017208
Data columns (total 24 columns):
 #   Column                     Non-Null Count    Dtype         
---  ------                     --------------    -----         
 0   Store                      1017209 non-null  int64         
 1   DayOfWeek                  1017209 non-null  int64         
 2   Date                       1017209 non-null  datetime64[us]
 3   Sales                      1017209 non-null  int64         
 4   Customers                  1017209 non-null  int64         
 5   Open                       1017209 non-null  int64         
 6   Promo                      1017209 non-null  int64         
 7   StateHoliday               1017209 non-null  object        
 8   SchoolHoliday              1017209 non-null  int64         
 9   Month                      1017209 non-null  int32         
 10  Year                       1017209 non-null  int32         
 11  Day                        1017209 non-null  int

In [40]:
#check if there's any str data type left in the train data
train.select_dtypes(include=['object', 'string']).columns

Index(['StateHoliday', 'PromoInterval'], dtype='str')

In [41]:
#do one-hot encoding for the 'StateHoliday' and 'PromoInterval' columns
train = pd.get_dummies(train, columns=['StateHoliday', 'PromoInterval'], drop_first=True)

In [42]:
train.select_dtypes(include=['object', 'string']).columns

Index([], dtype='str')

#### Now the data is ready

### Modeling Prepraion